# House Stage 1 regression

Model specs (estimator, base predictors, target): `experiments/house/configs.py`. Feature sets and holdout years: `src/house/regression/config.py`. Columns: `src/house/regression/schema.py`.
Build the predictors and targets first: `python data-processing/house/house_features.py` (`modeling.csv`) and `python data-processing/house/house_targets.py` (`targets.csv`).
Experiments run through `experiments.house.run.run_experiment`, the same function as `python -m experiments.house.run` (see its docstring): years, training inputs (`non_imputed` / `all`), and `features` / `add_features` / `drop_features` are arguments. It joins the config's target onto the predictors, checks that the keys match, and only then selects years and fits. Nothing is saved unless you ask (`save=True` or `tracking.save_run`).

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path

import matplotlib.pyplot as plt

sys.path.insert(0, str(Path("../../src").resolve()))
sys.path.insert(0, str(Path("../..").resolve()))  # repository root, for experiments.house
import house.regression as hr  # hr.<Tab> lists everything; names are in src/house/regression/__init__.py
from experiments.house import tracking
from experiments.house.run import run_experiment

In [ ]:
predictors = hr.load_predictors()  # data/processed/house/modeling.csv, every year (2006-2026)
targets = hr.load_targets()        # data/processed/house/targets.csv, every year (2006-2024)
hr.validate_schema(predictors).issues

In [ ]:
hr.validate_schema(targets, schema=hr.TARGETS_SCHEMA).issues

In [ ]:
YEARS = {"train_years": list(range(2006, 2022, 2)), "eval_years": [2022]}  # predict_years=[2026] for a forecast
DATA = {"predictors": predictors, "targets": targets}  # reuse the loaded tables instead of reloading per run

result = run_experiment(model="classic_ols", **YEARS, **DATA)  # save=False by default
print(hr.describe_config(result.config))

In [ ]:
joined = hr.join_target(predictors, targets, result.config["target"])  # what run_experiment selects years from
joined.groupby("year")[result.config["target"]].count()

In [ ]:
result.model.results.summary()

In [ ]:
result.metrics  # "all" targets include modeled imputations; "non_imputed" are observed only

## Prediction diagnostics

In [ ]:
scored = result.predictions[result.predictions["role"] == "eval"].dropna(subset=["actual"])
scored.groupby("imputed")["residual"].describe()  # residual = actual - prediction

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5.5))
for imputed, color, label in [(False, "#2a78d6", "observed"), (True, "#eb6834", "modeled imputation")]:
    rows = scored[scored["imputed"] == imputed]
    ax.scatter(rows["prediction"], rows["actual"], s=14, color=color, alpha=0.7, linewidths=0, label=label)
lim = [scored[["prediction", "actual"]].min().min(), scored[["prediction", "actual"]].max().max()]
ax.plot(lim, lim, color="#888", lw=1, zorder=0)
ax.set(xlabel="predicted margin (D +)", ylabel="actual margin (D +)", title=f"{result.config['model']}: actual vs. predicted")
ax.grid(alpha=0.2)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
plt.show()

## Ablations and variants (not saved)

Every run here uses the same evaluation years, so `compare` checks `same_eval_rows` / `same_train_rows`: a metric change is a modeling change, not a sample change.

In [ ]:
variants = [run_experiment(model="classic_ols", **YEARS, **DATA, drop_features=[f], name=f"no_{f}")
            for f in result.config["features"]]
variants += [run_experiment(model="classic_ols", **YEARS, **DATA, add_features=["college", "hispanic"], name="plus_acs"),
             run_experiment(model="classic_ols", **YEARS, **DATA, training_inputs="all", name="train_all")]
tracking.compare([result, *variants])

In [ ]:
tracking.compare_predictions([result, *variants]).head()  # common evaluation rows, one column per run

## Saved runs

Save the runs worth keeping: `tracking.save_run(result)` or `run_experiment(..., save=True)`. Each gets a folder in `results/house/runs/` and a row in `results/house/runs.csv`; the CLI saves by default.

In [ ]:
# run_id = tracking.save_run(result)

In [ ]:
runs = tracking.load_runs()  # after at least one saved run
runs.sort_values("mae_non_imputed")[["run_id", "features", "training_inputs", "n_train", "mae_non_imputed",
                                      "rmse_non_imputed", "bias_non_imputed", "mae_all"]]

In [ ]:
saved = tracking.load_run(runs["run_id"].iloc[-1])  # config, summary, metrics, coefficients, predictions
saved.coefficients

In [ ]:
tracking.compare(runs["run_id"].tail(3).tolist())